#### 1. CONFIGURACIÓN E IMPORTACIONES

In [1]:
print("--- 1. Importando librerías ---")
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import joblib
import optuna

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

print("Librerías importadas y configuración completa.\n")

--- 1. Importando librerías ---
Librerías importadas y configuración completa.



c:\Users\Omar\Desktop\trabajo\repositorios\p-x-nlp-feel-recognize\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 2. CARGA DE DATOS Y PREPROCESAMIENTO

In [2]:
print("--- 2. Cargando datos y aplicando preprocesamiento ---")
# Usamos exactamente el mismo preprocesamiento para garantizar la consistencia
file_path = '../data/youtube_comment_dataset.csv'
df = pd.read_csv(file_path)
df.dropna(subset=['Text'], inplace=True)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    text = str(text).lower(); text = re.sub(r'https?://\S+|www\.\S+', '', text); text = re.sub(r'<.*?>+', '', text); text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split(); clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(clean_tokens)
df['Processed_Text'] = df['Text'].apply(preprocess_text)
label_cols = [col for col in df.columns if col.startswith('Is')]; [df.__setitem__(col, df[col].apply(lambda x: 1 if str(x).upper() == 'TRUE' else 0)) for col in label_cols]
df['IsHate'] = df[label_cols].any(axis=1).astype(int)
print("Carga y preprocesamiento consistentes aplicados.\n")

# Dividir los datos (usamos la misma división que antes para comparar)
X = df['Processed_Text']
y = df['IsHate']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

--- 2. Cargando datos y aplicando preprocesamiento ---
Carga y preprocesamiento consistentes aplicados.



C:\Users\Omar\AppData\Local\Temp\ipykernel_7356\2944582989.py:13: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  label_cols = [col for col in df.columns if col.startswith('Is')]; [df.__setitem__(col, df[col].apply(lambda x: 1 if str(x).upper(

#### 3. OPTIMIZACIÓN DEL MODELO XGBOOST CON OPTUNA

In [3]:
print("--- 3. Optimizando el pipeline de XGBoost ---")

# La función 'objective' es lo que Optuna intentará maximizar.
# Recibe un 'trial' que sugiere los hiperparámetros.
def objective_xgb(trial):
    # Definimos el espacio de búsqueda de hiperparámetros
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
    }

    # Creamos el pipeline dentro de la función objective
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('clf', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', **params))
    ])
    
    # Usamos validación cruzada para una evaluación más robusta
    # Esto evita sobreajustar a una única división de validación
    score = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='f1', n_jobs=-1).mean()
    
    return score

# Creamos el estudio de Optuna y lo ejecutamos
# La dirección es 'maximize' porque queremos maximizar el F1-score
study_xgb = optuna.create_study(direction='maximize')
# n_trials es el número de combinaciones de hiperparámetros que probará
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True) 

print("\nOptimización de XGBoost completada.")
print(f"Mejor F1-score (validación cruzada): {study_xgb.best_value:.4f}")
print("Mejores hiperparámetros encontrados:")
print(study_xgb.best_params)

[I 2025-07-14 18:26:24,669] A new study created in memory with name: no-name-53c6f706-400b-4e99-861d-f44d5cdc14bf


--- 3. Optimizando el pipeline de XGBoost ---


Best trial: 0. Best value: 0.634107:   2%|▏         | 1/50 [00:02<02:08,  2.61s/it]

[I 2025-07-14 18:26:27,283] Trial 0 finished with value: 0.6341068022886205 and parameters: {'n_estimators': 573, 'max_depth': 5, 'learning_rate': 0.04564326067702339, 'subsample': 0.7040959867769198, 'colsample_bytree': 0.9467296554863998, 'gamma': 0.7614948061902632}. Best is trial 0 with value: 0.6341068022886205.


Best trial: 0. Best value: 0.634107:   4%|▍         | 2/50 [00:03<01:29,  1.86s/it]

...

[I 2025-07-14 18:27:01,059] Trial 46 finished with value: 0.6459545319957094 and parameters: {'n_estimators': 402, 'max_depth': 5, 'learning_rate': 0.2851252025470828, 'subsample': 0.7736563820301228, 'colsample_bytree': 0.7596214049637169, 'gamma': 0.23555509467668834}. Best is trial 46 with value: 0.6459545319957094.
[I 2025-07-14 18:27:01,253] Trial 47 finished with value: 0.5858854689180859 and parameters: {'n_estimators': 281, 'max_depth': 5, 'learning_rate': 0.28064935082905385, 'subsample': 0.7388158268417523, 'colsample_bytree': 0.8101552178964946, 'gamma': 3.465386667617929}. Best is trial 46 with value: 0.6459545319957094.


Best trial: 46. Best value: 0.645955:  98%|█████████▊| 49/50 [00:36<00:00,  2.46it/s]

[I 2025-07-14 18:27:01,519] Trial 48 finished with value: 0.5567019931642175 and parameters: {'n_estimators': 411, 'max_depth': 4, 'learning_rate': 0.08342852819996982, 'subsample': 0.7921196476120027, 'colsample_bytree': 0.7190447352884416, 'gamma': 4.738104179466775}. Best is trial 46 with value: 0.6459545319957094.


Best trial: 46. Best value: 0.645955: 100%|██████████| 50/50 [00:37<00:00,  1.34it/s]

[I 2025-07-14 18:27:01,900] Trial 49 finished with value: 0.6459074186673545 and parameters: {'n_estimators': 488, 'max_depth': 5, 'learning_rate': 0.2597126955214305, 'subsample': 0.9959587014470785, 'colsample_bytree': 0.8246387700094743, 'gamma': 0.6822723000192296}. Best is trial 46 with value: 0.6459545319957094.

Optimización de XGBoost completada.
Mejor F1-score (validación cruzada): 0.6460
Mejores hiperparámetros encontrados:
{'n_estimators': 402, 'max_depth': 5, 'learning_rate': 0.2851252025470828, 'subsample': 0.7736563820301228, 'colsample_bytree': 0.7596214049637169, 'gamma': 0.23555509467668834}


#### 4. ENTRENAR Y EVALUAR EL MODELO XGBOOST OPTIMIZADO

In [4]:
print("\n--- 4. Entrenando el modelo XGBoost final con los mejores hiperparámetros ---")

# Obtenemos los mejores hiperparámetros
best_params_xgb = study_xgb.best_params

# Creamos el pipeline final con esos parámetros
pipeline_xgb_optimized = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', **best_params_xgb))
])

# Entrenamos en TODO el conjunto de entrenamiento
pipeline_xgb_optimized.fit(X_train, y_train)

# Evaluamos en el conjunto de prueba (datos nunca vistos)
y_pred_optimized = pipeline_xgb_optimized.predict(X_test)
print("\nReporte de Clasificación (XGBoost OPTIMIZADO en Test):")
print(classification_report(y_test, y_pred_optimized, target_names=['No Odio', 'Odio']))


--- 4. Entrenando el modelo XGBoost final con los mejores hiperparámetros ---


c:\Users\Omar\Desktop\trabajo\repositorios\p-x-nlp-feel-recognize\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:27:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Reporte de Clasificación (XGBoost OPTIMIZADO en Test):
              precision    recall  f1-score   support

     No Odio       0.69      0.80      0.74       108
        Odio       0.71      0.59      0.64        92

    accuracy                           0.70       200
   macro avg       0.70      0.69      0.69       200
weighted avg       0.70      0.70      0.70       200



#### 5. COMPARACIÓN FINAL Y GUARDADO DEL NUEVO CAMPEÓN

In [7]:
from sklearn.metrics import f1_score

print("\n--- 5. Comparación final y guardado del modelo ---")

# 5.1 Cargar el modelo campeón anterior desde el archivo .joblib
# ¡Aquí es donde usamos el resultado del notebook anterior!
print("Cargando el modelo campeón anterior 'logreg_pipeline.joblib' para comparación...")
try:
    champion_anterior = joblib.load('logreg_pipeline.joblib')
except FileNotFoundError:
    print("ERROR: No se encontró 'logreg_pipeline.joblib'. Asegúrate de que está en la carpeta 'model/'.")
    # Si no lo encuentra, creamos uno para que el código no falle y puedas depurar.
    from sklearn.linear_model import LogisticRegression
    champion_anterior = Pipeline([('clf', LogisticRegression())]) # Modelo dummy
    champion_anterior.fit(X_train, y_train)


# 5.2 Evaluar el rendimiento del campeón anterior en el conjunto de prueba
print("Evaluando el rendimiento del modelo anterior...")
y_pred_anterior = champion_anterior.predict(X_test)
f1_anterior = f1_score(y_test, y_pred_anterior)

# 5.3 Obtener el rendimiento del nuevo modelo optimizado (que ya calculamos antes)
y_pred_nuevo = pipeline_xgb_optimized.predict(X_test)
f1_nuevo = f1_score(y_test, y_pred_nuevo)


# 5.4 Comparar los resultados cara a cara
print("\n" + "="*30)
print("     COMPARACIÓN FINAL DE F1-SCORE")
print("="*30)
print(f"  - Modelo Anterior (Logistic Regression): {f1_anterior:.4f}")
print(f"  - Modelo Nuevo (XGBoost Optimizado):   {f1_nuevo:.4f}")
print("="*30)


# 5.5 Decidir el nuevo campeón y guardar
if f1_nuevo > f1_anterior:
    print("\n¡El XGBoost optimizado es el NUEVO CAMPEÓN!")
    print("Ha superado al modelo de Regresión Logística.")
    joblib.dump(pipeline_xgb_optimized, 'optimized_xgb_pipeline.joblib')
    print("Nuevo modelo guardado como 'optimized_xgb_pipeline.joblib'")
else:
    print("\nEl modelo de Regresión Logística SIGUE SIENDO EL CAMPEÓN.")
    print("La optimización no logró superar al baseline. El modelo anterior es el mejor hasta ahora.")


--- 5. Comparación final y guardado del modelo ---
Cargando el modelo campeón anterior 'logreg_pipeline.joblib' para comparación...
Evaluando el rendimiento del modelo anterior...

     COMPARACIÓN FINAL DE F1-SCORE
  - Modelo Anterior (Logistic Regression): 0.6816
  - Modelo Nuevo (XGBoost Optimizado):   0.6429

El modelo de Regresión Logística SIGUE SIENDO EL CAMPEÓN.
La optimización no logró superar al baseline. El modelo anterior es el mejor hasta ahora.
